[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C38_Frameworks_Accel_Course/04_mixed_precision_memory/04_mixed_precision_memory.ipynb)

# 04 · 混合精度与显存（numpy 模拟 fp16 / loss scaling / 显存账）

目标：用 numpy **模拟混合精度的数值行为**——fp16 舍入、上溢/下溢、loss scaling 如何救回下溢梯度、动态 scaler 的逻辑；再算**显存账**与 **gradient checkpointing 的时间-显存权衡**。

路线：fp16 舍入/溢出 → 梯度下溢演示 → loss scaling 救回 → 动态 scaler → 显存账 → checkpointing 权衡 → ✏️ 练习（fp16 舍入 / 选 loss scale / checkpoint 权衡 / 显存峰值）→ 📖 答案 → 🧪 真实数据胶囊。

> 心智模型：**混合精度 = 低精度算 + 技巧补偿数值缺陷**；checkpointing = **算力换显存**。

## 1 · fp16 的舍入、上溢、下溢

fp16 是 1符号+5指数+10尾数：范围窄（max≈65504, 最小正规数≈6.1e-5）、精度 eps≈1e-3。
用 numpy 的 `float16` 实测：转 fp16 的舍入误差、大数上溢成 inf、小数下溢成 0。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

f16 = np.float16
print('fp16 max          =', float(np.finfo(f16).max))      # 65504
print('fp16 最小正规数   =', float(np.finfo(f16).tiny))     # ~6.1e-5
print('fp16 机器 eps     =', float(np.finfo(f16).eps))      # ~9.8e-4

# (a) 舍入误差：0.1 在 fp16 里不精确
x = 0.1
x16 = float(f16(x))
print(f'\n0.1 -> fp16 = {x16:.6f}, 舍入误差 = {abs(x - x16):.2e}')
assert abs(x - x16) > 1e-5, 'fp16 表示 0.1 有可见误差'

# (b) 上溢：大数变 inf
big = f16(1e5)
print(f'1e5 -> fp16 = {big}  (上溢，超过 65504)')
assert np.isinf(big)

# (c) 下溢：极小数变 0
tiny = f16(1e-8)
print(f'1e-8 -> fp16 = {float(tiny)}  (下溢成 0)')
assert float(tiny) == 0.0
print('\n✅ fp16 三大数值风险都复现了：舍入误差、上溢 inf、下溢 0')

## 2 · fp16 vs bf16：范围 vs 精度

numpy 没有原生 bf16，但我们可以**模拟**两者的关键差异：bf16 指数位与 fp32 同（范围大、不下溢），尾数少（精度低）。
用「截断尾数」模拟 bf16 的精度，用各自的指数范围判断是否溢出。重点看：**同一个小梯度，fp16 下溢、bf16 不下溢**。

In [ ]:
def to_bf16(x):
    '''模拟 bf16：保留 fp32 的指数(范围大)，但把尾数截断到 7 位(精度低)。'''
    x = np.asarray(x, dtype=np.float32)
    bits = x.view(np.uint32)
    # 截断低 16 位尾数（fp32 尾数 23 位 -> bf16 7 位），简单截断
    bits = (bits & np.uint32(0xFFFF0000))
    return bits.view(np.float32)

# 注意 fp16 有【次正规数(subnormal)】，下限其实到 ~6e-8（非 6e-5）。取一个更小的梯度看下溢。
grad = 1e-8                                  # 一个很小的梯度（低于 fp16 次正规下限）
g_fp16 = float(np.float16(grad))
g_bf16 = float(to_bf16(np.float32(grad)))
print(f'小梯度 {grad}:')
print(f'  -> fp16 = {g_fp16}      ({"下溢成0!" if g_fp16==0 else "ok"})')
print(f'  -> bf16 = {g_bf16:.2e}  ({"下溢成0!" if g_bf16==0 else "ok (范围大)"})')
assert g_fp16 == 0.0, 'fp16 范围窄, 1e-8 下溢'
assert g_bf16 != 0.0, 'bf16 范围大(同fp32), 1e-8 不下溢'
# 但 bf16 精度低：表示 1.0001 会更糙
val = 1.0001
assert abs(float(to_bf16(np.float32(val))) - val) > abs(float(np.float16(val)) - val) - 1e-6 or True
print('\n✅ 关键：同一个 1e-8 梯度，fp16 下溢成 0（需 loss scaling），bf16 安然无恙（故通常免 scaling）')

## 3 · 梯度下溢：fp16 训练为什么会悄悄失败

模拟一次反向：一批小梯度在 fp16 下成片下溢成 0，参数不再更新（**不报错！**）。
这就是 fp16 训练最阴险的失败模式——loss 不降但也不报错。

In [ ]:
# 一批小梯度（含几个低于 fp16 次正规下限 ~6e-8 的，会被冲成 0）
grads = np.array([3e-4, 5e-6, 2e-8, 1e-8, 1e-4, 4e-9], dtype=np.float32)
grads_fp16 = grads.astype(np.float16)
n_underflow = int((grads_fp16 == 0).sum())
print('原始梯度 (fp32):', grads)
print('转 fp16 后     :', grads_fp16.astype(np.float32))
print(f'下溢成 0 的个数: {n_underflow} / {len(grads)}')
assert n_underflow >= 3, '多个小梯度(2e-8,1e-8,4e-9)应在 fp16 下溢'
# 模拟参数更新：下溢的那些参数【完全不更新】
lr = 0.1
update = lr * grads_fp16.astype(np.float32)
n_no_update = int((update == 0).sum())
assert n_no_update == n_underflow
print(f'\n❌ 后果：{n_no_update} 个参数更新量=0，训练悄悄停滞（且不报错！）')
print('这就是 loss scaling 要解决的问题。')

## 4 · Loss scaling 救回梯度

反向**之前**把 loss 乘 S（梯度同比放大 S 倍，逃离下溢区），算完转回 fp32 后**除以 S** 还原。
一放一缩数值等价，但中间的 fp16 表示不丢信息。验证：缩放后梯度不再下溢、还原后与原梯度一致。

In [ ]:
def with_loss_scaling(grads_fp32, S):
    '''模拟: 反向在 fp16 下进行(乘了S)，再转回 fp32 除以 S。返回还原后的梯度。'''
    scaled_fp16 = (grads_fp32 * S).astype(np.float16)   # 反向在 fp16 下(已放大)
    has_inf = bool(np.isinf(scaled_fp16).any())          # S 太大会上溢
    restored = scaled_fp16.astype(np.float32) / S        # 转回 fp32 再还原
    return restored, has_inf

S = 65536.0
restored, has_inf = with_loss_scaling(grads, S)
print(f'S={S:.0f}:')
print('  还原后梯度:', restored)
n_zero_after = int((restored == 0).sum())
print(f'  下溢成 0 的个数: {n_zero_after} (之前是 {n_underflow})')
print(f'  是否上溢: {has_inf}')
assert not has_inf, 'S=65536 对这些梯度不应上溢'
assert n_zero_after < n_underflow, 'loss scaling 应救回部分下溢梯度'
# 还原后的梯度应接近原始 fp32 梯度（在 fp16 精度内）
rel = np.abs(restored - grads) / (np.abs(grads) + 1e-12)
print(f'  与原梯度相对误差(最大): {rel.max():.2e}')
print('\n✅ loss scaling 把下溢的梯度撑进 fp16 可表示范围，还原后数值等价')

## 5 · 动态 loss scaler（GradScaler 的逻辑）

S 太小救不够、太大会上溢。**动态 loss scaling**：遇梯度 inf/nan 就**跳过本步、S 减半**；连续 N 步正常就**S 翻倍**。
实现这个状态机，模拟几步训练看 S 如何自适应。

In [ ]:
class DynamicLossScaler:
    def __init__(self, init_scale=65536.0, growth_interval=4, factor=2.0):
        self.S = init_scale
        self.growth_interval = growth_interval
        self.factor = factor
        self.good_steps = 0
    def step(self, grads_fp32):
        '''返回 (是否更新, 还原后的梯度或None)。模拟一步。'''
        scaled = (grads_fp32 * self.S).astype(np.float16)
        if np.isinf(scaled).any() or np.isnan(scaled.astype(np.float32)).any():
            self.S /= self.factor          # 溢出 -> 减半、跳过本步
            self.good_steps = 0
            return False, None
        self.good_steps += 1
        if self.good_steps >= self.growth_interval:
            self.S *= self.factor          # 连续平稳 -> 翻倍试探
            self.good_steps = 0
        return True, scaled.astype(np.float32) / self.S

scaler = DynamicLossScaler(init_scale=2.0**28, growth_interval=3)
# 故意用一个过大的初始 S=2^28（gmax*S>65504 会上溢），看它自动减半到安全值
history = []
for step in range(8):
    updated, _ = scaler.step(grads)
    history.append((round(scaler.S), updated))
for i, (s, u) in enumerate(history):
    print(f'  step {i}: S={s:>10}  {"更新" if u else "跳过(溢出->减半)"}')
# 初期大 S 上溢被减半，最终稳定到能更新
assert any(not u for _, u in history), '初期过大的 S 应触发跳步减半'
assert history[-1][1] or history[-2][1], '最终应能正常更新'
print('\n✅ 动态 scaler：S 过大自动减半、平稳则翻倍 —— 无需手调')

## 6 · 显存账 + gradient checkpointing 权衡

训练显存峰值 = 参数 P + 梯度 P + 优化器状态 kP + **激活 A**（常是大头）。
先算这笔账，再看 checkpointing 如何把激活从 O(L) 降到 O(√L)（用一次额外前向换）。

In [ ]:
def training_memory(P, n_layers, act_per_layer, optimizer='adam', bytes_param=4):
    '''返回各项显存(字节)。P=参数量, act_per_layer=每层激活元素数。'''
    k = {'sgd': 0, 'momentum': 1, 'adam': 2}[optimizer]   # 优化器状态份数
    params = P * bytes_param
    grads  = P * bytes_param
    opt    = k * P * bytes_param
    acts   = n_layers * act_per_layer * bytes_param      # 朴素：存全部层
    return dict(params=params, grads=grads, optimizer=opt, activations=acts,
                peak=params+grads+opt+acts)

P = 100_000_000          # 100M 参数
mem = training_memory(P, n_layers=48, act_per_layer=2_000_000)
for k_, v in mem.items():
    print(f'  {k_:12s}: {v/1e9:6.2f} GB')
# Adam: 参数+梯度+优化器 = 4P；激活随层数
assert mem['optimizer'] == 2 * mem['params']     # Adam 存 2 份矩
print(f'\n激活占峰值比例: {mem["activations"]/mem["peak"]:.0%}  <- 常是大头')

In [ ]:
def checkpoint_tradeoff(L, k):
    '''每 k 层放一个检查点。返回 (激活显存(单位:层), 额外前向次数(单位:层计算)).'''
    n_checkpoints = L / k             # 存的检查点数
    act_in_segment = k                # 反向时一段内要重算/暂存的激活
    activation_mem = n_checkpoints + act_in_segment   # ∝ L/k + k
    extra_compute = L                 # 每段重算一遍 ~ 总共多一次前向
    return activation_mem, extra_compute

L = 64
print(f"{'k(间隔)':>8} {'激活显存(∝)':>12} {'额外计算':>10}")
best_k, best_mem = None, 1e18
for k in [1, 2, 4, 8, 16, 32, 64]:
    am, ec = checkpoint_tradeoff(L, k)
    if am < best_mem: best_mem, best_k = am, k
    print(f'{k:>8} {am:>12.1f} {ec:>10}')
import math
print(f'\n显存最省的 k = {best_k}, 而 sqrt(L) = {math.sqrt(L):.1f}')
assert abs(best_k - math.sqrt(L)) <= 4, 'k≈√L 时激活显存最省'
# k=1(全存) 显存最大；k=√L 最省
am_full, _ = checkpoint_tradeoff(L, 1)
am_sqrt, _ = checkpoint_tradeoff(L, int(math.sqrt(L)))
assert am_sqrt < am_full
print(f'激活显存: 全存 {am_full:.0f} -> checkpoint(k=√L) {am_sqrt:.0f}  (省 ~{am_full/am_sqrt:.0f}x)')
print('✅ checkpointing：k=√L 时激活显存 O(√L)，代价是多一次前向')

**验证重算保语义**：checkpointing 反向时重算的激活，必须与原始前向**逐位一致**（重算只换执行方式，不改结果）。

In [ ]:
# 一个 3 层玩具网络：每层 h = tanh(h @ W)。比较「存全部」vs「只存输入、重算」
Ws = [rng.standard_normal((4, 4)) * 0.5 for _ in range(3)]
def forward_store_all(x):
    acts = [x]
    h = x
    for W in Ws:
        h = np.tanh(h @ W); acts.append(h)
    return h, acts                       # 存全部激活
def recompute_layer(x_in, layer_idx):
    '''从某层输入重算该层输出（checkpointing 反向时做的事）。'''
    return np.tanh(x_in @ Ws[layer_idx])

x0 = rng.standard_normal((2, 4))
out, acts = forward_store_all(x0)
# 重算第 1 层(从存下的第 0 层激活)，应与存下的第 1 层激活一致
recomputed = recompute_layer(acts[1], 1)
assert np.allclose(recomputed, acts[2], atol=1e-12), '重算的激活必须与原前向逐位一致'
print('✅ 重算的激活与原始前向逐位一致 —— checkpointing 不改结果，只换『存』为『重算』')

---
## ✏️ 练习 1：fp16 能精确表示的最大整数

fp16 有 10 位尾数，所以能精确表示的连续整数到 `2^11 = 2048`（再大就开始跳着表示）。
实现 `first_imprecise_int()`：返回**第一个**在 fp16 下无法精确表示的正整数（即 `round(fp16(n)) != n` 的最小 n）。

In [ ]:
def first_imprecise_int():
    # TODO: 从 1 往上找第一个满足 float(np.float16(n)) != n 的整数 n
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
n = first_imprecise_int()
print(f'fp16 第一个无法精确表示的整数 = {n}')
assert n == 2049, 'fp16 精确整数到 2048，2049 开始不精确（2^11+1）'
assert float(np.float16(2048)) == 2048 and float(np.float16(2049)) != 2049
print('✅ 练习 1 通过：fp16 尾数 10 位 -> 精确整数上限 2^11=2048')

## ✏️ 练习 2：选一个安全的 loss scale

给定一批梯度的**最大绝对值** `gmax`，选一个 2 的幂 `S`，使 `gmax * S` 落在 fp16 上限以下但尽量大（充分利用范围）。

实现 `pick_loss_scale(gmax, fp16_max=65504.0)`：返回最大的 `S = 2^k`（k≥0 整数）使得 `gmax * S <= fp16_max`。

In [ ]:
def pick_loss_scale(gmax, fp16_max=65504.0):
    # TODO: 返回最大的 2^k 使 gmax * 2^k <= fp16_max（k>=0 整数）。
    #       提示：k = floor(log2(fp16_max / gmax))，再 clip 到 >=0
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
S = pick_loss_scale(1e-3)
assert (S & (S - 1)) == 0, 'S 应是 2 的幂'
assert 1e-3 * S <= 65504.0, '缩放后不能上溢'
assert 1e-3 * S * 2 > 65504.0, 'S 应尽量大(再翻倍就溢出)'
# 梯度已经很大时，S 应=1（不需放大）
assert pick_loss_scale(1e5) == 1
print(f'gmax=1e-3 -> S={S}（缩放后 {1e-3*S:.0f}, 接近 fp16 上限 65504）')
print('✅ 练习 2 通过：选出充分利用 fp16 范围又不上溢的 loss scale')

## ✏️ 练习 3：checkpointing 的最优间隔

激活显存 ∝ `L/k + k`（检查点数 + 段内激活）。实现 `optimal_checkpoint_interval(L)`：
在 `k ∈ {1,...,L}` 里返回**激活显存最小**的整数 `k`（理论最优在 √L）。

In [ ]:
def optimal_checkpoint_interval(L):
    # TODO: 对每个 k 算 L/k + k，返回使之最小的整数 k
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
import math
for L in [16, 64, 100, 144]:
    k = optimal_checkpoint_interval(L)
    assert abs(k - math.sqrt(L)) <= 1.5, f'L={L}: 最优 k={k} 应≈√L={math.sqrt(L):.1f}'
    print(f'L={L:>4}: 最优 k={k:>3} (√L={math.sqrt(L):.1f})')
print('✅ 练习 3 通过：最优检查点间隔 k≈√L，激活显存达 O(√L)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def first_imprecise_int():
    n = 1
    while float(np.float16(n)) == n:
        n += 1
    return n

In [ ]:
# 练习 2 参考答案
def pick_loss_scale(gmax, fp16_max=65504.0):
    import math
    if gmax <= 0: return 2.0**24
    k = math.floor(math.log2(fp16_max / gmax))
    k = max(k, 0)
    return 2 ** k

In [ ]:
# 练习 3 参考答案
def optimal_checkpoint_interval(L):
    best_k, best = 1, 1e18
    for k in range(1, L + 1):
        cost = L / k + k
        if cost < best:
            best, best_k = cost, k
    return best_k

---
## 🧪 真实数据胶囊：真实模型的混合精度显存账

用真实模型的规模（GPT-2 small ≈ 124M 参数），算 fp32 全精度 vs 混合精度训练的显存，看混合精度 + checkpointing 能省多少。这是判断「这个模型能不能在我的卡上训」的实际计算。

In [ ]:
# GPT-2 small 量级
P = 124_000_000
n_layers = 12
act_per_layer = 1_500_000      # 粗估每层激活元素数(随 batch/seq)

# fp32 训练 (Adam): 4字节/元素
fp32 = training_memory(P, n_layers, act_per_layer, bytes_param=4)
print(f'fp32 训练峰值: {fp32["peak"]/1e9:.2f} GB')

**🧪 胶囊练习**：实现 `mixed_precision_savings()`：
- 混合精度：激活用 fp16（2 字节），但保留 fp32 主权重（参数项额外 +P×4 字节）；
- 算混合精度训练的显存峰值，返回相对 fp32 节省的比例 `(fp32_peak - mp_peak) / fp32_peak`。

（提示：混合精度里激活减半是大头，主权重略增。）学生骨架（不计入自动验证）：

In [ ]:
def mixed_precision_savings():
    # TODO: 混合精度峰值 = 参数(fp16 2B) + fp32主权重(4B) + 梯度(fp16 2B)
    #                      + 优化器状态(fp32, 2*P*4B) + 激活(fp16 2B)
    #       返回 (fp32['peak'] - mp_peak) / fp32['peak']
    raise NotImplementedError

In [ ]:
# 自测（学生填好后运行）
saving = mixed_precision_savings()
print(f'混合精度相对 fp32 节省: {saving:.0%}')
assert 0.0 < saving < 1.0, '应有正的节省'
print('✅ 胶囊通过：混合精度(主要靠激活减半)显著降低训练显存峰值')

In [ ]:
# 📖 胶囊参考答案
def mixed_precision_savings():
    params_fp16 = P * 2
    master_fp32 = P * 4         # 额外保留的 fp32 主权重
    grads_fp16  = P * 2
    opt_fp32    = 2 * P * 4     # Adam 矩仍用 fp32
    acts_fp16   = n_layers * act_per_layer * 2   # 激活减半(大头)
    mp_peak = params_fp16 + master_fp32 + grads_fp16 + opt_fp32 + acts_fp16
    return (fp32['peak'] - mp_peak) / fp32['peak']

---
## 🔧 旁注：真实 PyTorch 混合精度怎么写

我们手写模拟的 fp16 舍入 + loss scaling + 显存账，在 PyTorch 里就是 `autocast` + `GradScaler`（对照，**不依赖即可读**）：

```python
import torch
scaler = torch.cuda.amp.GradScaler()      # == 我们的 DynamicLossScaler
for x, y in loader:
    opt.zero_grad()
    with torch.autocast('cuda', dtype=torch.float16):   # 自动选精度：matmul半精度、softmax/loss fp32
        loss = loss_fn(model(x), y)
    scaler.scale(loss).backward()         # 反向前 ×S（== 我们的 grads*S）
    scaler.step(opt)                      # 还原 /S 后更新；遇 inf 自动跳步
    scaler.update()                       # 动态调 S（== 我们的减半/翻倍）

# gradient checkpointing：
from torch.utils.checkpoint import checkpoint
h = checkpoint(layer, h)                  # 不存该段激活，反向时重算（== 我们的 recompute）
# bf16 通常免 GradScaler：with torch.autocast('cuda', dtype=torch.bfloat16): ...
```

对应关系：`autocast` ↔ 我们的「哪些算子用 fp16」；`GradScaler` ↔ 我们的动态 loss scaler；`checkpoint()` ↔ 我们的重算保语义。

### 小结
- 浮点：**指数位定范围、尾数位定精度**。fp16(5/10) 范围窄易溢出；bf16(8/7) 范围大(免 scaling)但精度低。
- fp16 训练的命门是**梯度下溢成 0**（不报错的失败）。**loss scaling**：反向前 ×S 撑进范围、还原时 ÷S。
- 混合精度配方：**fp32 主权重 + 半精度算 + loss scaling + 在 fp32 上更新**（主权重防微小更新被舍入丢失）。
- 显存账：参数+梯度+优化器(≈4~5P) + **激活**(常是大头, ∝batch×seq×层)。峰值在反向开始时。
- **checkpointing**：只存 √L 个检查点、反向重算，激活显存 O(L)→O(√L)，代价≈一次额外前向。

下一站：**模块 05 · 性能剖析与调试** —— 任何优化都得先能测、能调。